## PMXT API -> Kafka 原始数据链路测试

本部分用于验证当前目标的数据流前半段：

- 从 PMXT API 获取 `fetch_markets` 原始数据
- 以 JSON 消息写入 Kafka
- 从 Kafka 消费并检查消息结构


In [1]:
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd

# 让 notebook 里可以 import repo 内的 app 包
repo_root = Path.cwd()
repo_root = repo_root.parent
if (repo_root / "app").exists():
    sys.path.insert(0, str(repo_root))

from app.config import settings as app_settings
from app.clients.kafka_client import KafkaConsumerClient

print("Kafka bootstrap servers:", app_settings.KAFKA_BOOTSTRAP_SERVERS)
print("Kafka raw markets topic:", app_settings.KAFKA_TOPIC_POLYMARKET_MARKETS_RAW)


Kafka bootstrap servers: localhost:9092
Kafka raw markets topic: polymarket.markets.raw


In [8]:
from datetime import datetime

DO_INGEST = False  # 设为 True 才会请求 PMXT 并写入 Kafka
DO_CONSUME_CHECK = True  # 读取 Kafka 检查消息内容

if DO_INGEST:
    from app.services.ingestion_service import ingestion_service

    # 默认参数：sort='volume', limit=2000，但是这里测试只做5条
    published = ingestion_service.publish_polymarket_markets_to_kafka(
        # query="Trump",
        limit=5,
        sort="volume",
    )
    print("Published market messages to Kafka:", published)
else:
    print("Skip ingestion (DO_INGEST=False).")

if DO_CONSUME_CHECK:
    topic = app_settings.KAFKA_TOPIC_POLYMARKET_MARKETS_RAW
    group_id = f"notebook.kafka.check.markets.raw.{datetime.utcnow().strftime('%H%M%S')}"

    with KafkaConsumerClient(
        topic=topic,
        group_id=group_id,
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        consumer_timeout_ms=3000,
    ) as consumer:
        result = consumer.consume(max_messages_per_partition=5, max_workers=3)

    rows = []
    for partition, messages in result.items():
        for msg in messages:
            rows.append(
                {
                    "partition": partition,
                    "market_id": msg.get("market_id"),
                    "source": msg.get("source"),
                    "exchange": msg.get("exchange"),
                    "captured_at": msg.get("captured_at"),
                    "has_payload": "payload" in msg,
                    # "payload_keys": list((msg.get("payload") or {}).keys()),
                    'payload': msg.get("payload")
                }
            )

    print(f"Consumer group: {group_id}")
    print(f"Consumed messages: {len(rows)}")
    if rows:
        display(pd.DataFrame(rows).head(20))
    else:
        print("No messages consumed. Check broker/topic connectivity and try DO_INGEST=True.")


Skip ingestion (DO_INGEST=False).
2026-03-21 15:49:31 | INFO | app.clients.kafka_client | Initializing KafkaConsumerClient (topic=polymarket.markets.raw, group_id=notebook.kafka.check.markets.raw.074931)
2026-03-21 15:49:31 | INFO | app.clients.kafka_client | Opening Kafka consumer connection (topic=polymarket.markets.raw, group_id=notebook.kafka.check.markets.raw.074931)


C:\Users\Lily\AppData\Local\Temp\ipykernel_18988\3578496187.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  group_id = f"notebook.kafka.check.markets.raw.{datetime.utcnow().strftime('%H%M%S')}"


2026-03-21 15:49:31 | INFO | app.clients.kafka_client | Starting scheduled bounded consumption (topic=polymarket.markets.raw, partitions=[0, 1, 2], max_per_partition=5)
2026-03-21 15:49:31 | INFO | app.clients.kafka_client | Partition 0 bounded consume (start=0, stop=5, end=8)
2026-03-21 15:49:31 | INFO | app.clients.kafka_client | Partition 1 bounded consume (start=0, stop=2, end=2)
2026-03-21 15:49:31 | INFO | app.clients.kafka_client | Partition 2 bounded consume (start=0, stop=5, end=14)
2026-03-21 15:49:31 | INFO | app.clients.kafka_client | Partition 2 consume completed (5 messages)
2026-03-21 15:49:31 | INFO | app.clients.kafka_client | Partition 0 consume completed (5 messages)
2026-03-21 15:49:32 | INFO | app.clients.kafka_client | Partition 1 consume completed (2 messages)
2026-03-21 15:49:32 | INFO | app.clients.kafka_client | Scheduled bounded consumption completed (topic=polymarket.markets.raw, total_partitions=3, total_messages=12)
2026-03-21 15:49:32 | INFO | app.clients

,partition,market_id,source,exchange,captured_at,has_payload,payload
0,2,561973,pmxt,polymarket,2026-03-21T07:35:51.406617Z,True,"{'market_id': '561973', 'title': 'Republican P..."
1,2,561973,pmxt,polymarket,2026-03-21T07:36:44.250203Z,True,"{'market_id': '561973', 'title': 'Republican P..."
2,2,561973,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '561973', 'title': 'Republican P..."
3,2,562006,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '562006', 'title': 'Republican P..."
4,2,561998,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '561998', 'title': 'Republican P..."
5,0,561978,pmxt,polymarket,2026-03-21T07:35:51.406617Z,True,"{'market_id': '561978', 'title': 'Republican P..."
6,0,561978,pmxt,polymarket,2026-03-21T07:36:44.250203Z,True,"{'market_id': '561978', 'title': 'Republican P..."
7,0,561978,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '561978', 'title': 'Republican P..."
8,0,561263,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '561263', 'title': 'Presidential..."
9,0,561978,pmxt,polymarket,2026-03-21T07:45:04.589695Z,True,"{'market_id': '561978', 'title': 'Republican P..."


In [ ]:
# 单条数据示例
pd.DataFrame(rows)['payload'].iloc[0]

{'market_id': '561973',
 'title': 'Republican Presidential Nominee 2028 - Will Donald Trump win the 2028 Republican presidential nomination?',
 'outcomes': [{'outcome_id': '3039641309958397001906153616677074061284510636204155275446291716739429262374',
   'label': 'Donald Trump',
   'price': 0.0165,
   'price_change_24h': -0.0005,
   'metadata': {'clobTokenId': '3039641309958397001906153616677074061284510636204155275446291716739429262374'},
   'market_id': '561973'},
  {'outcome_id': '27828976648682466778776999076215423777766972981338264154049603024771135223200',
   'label': 'Not Donald Trump',
   'price': 0.9835,
   'price_change_24h': 0,
   'metadata': {'clobTokenId': '27828976648682466778776999076215423777766972981338264154049603024771135223200'},
   'market_id': '561973'}],
 'volume_24h': 26973.507839999995,
 'liquidity': 578147.26748,
 'url': 'https://polymarket.com/event/republican-presidential-nominee-2028',
 'description': 'This market will resolve to “Yes” if the named individu